In [ ]:
## 0 – Imports & basic configuration
import os, json, math, re
from pathlib import Path
from datetime import datetime
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from nltk.stem import WordNetLemmatizer
from discovery_utils.utils.llm import batch_check

try:
    from discovery_heat_pump_futures import PROJECT_DIR
except ModuleNotFoundError:
    PROJECT_DIR = Path(".").resolve()

load_dotenv()      


In [ ]:
## 1 – Load the full patent & paper datasets (no sampling!)
OPENALEX_URL = "https://discovery-hub-open-data.s3.eu-west-2.amazonaws.com/future_heat_pumps/heat_pumps_openalex.csv"
PATENT_URL   = "https://discovery-hub-open-data.s3.eu-west-2.amazonaws.com/future_heat_pumps/heat_pumps_patents.json"

openalex_df = pd.read_csv(OPENALEX_URL, low_memory=False)
patents_df  = pd.read_json(PATENT_URL, lines=True)

def format_title_abstract(df, title="title", abstract="abstract"):
    df = df.copy()
    df["title_abstract"] = (
        "TITLE: " + df[title].fillna("").str.lower() +
        " ABSTRACT: " + df[abstract].fillna("").str.lower()
    )
    return df

openalex_df = format_title_abstract(openalex_df)
patents_df  = format_title_abstract(patents_df)

print(f"Loaded {len(openalex_df):,} papers | {len(patents_df):,} patents")


In [ ]:
## 2 – Heat-pump taxonomy & keyword universe
CATEGORIES = {
    "1.1": "Compressors", "1.2": "Refrigerants", "1.3": "Heat-exchangers", "1.4": "Motor & Drives", "1.5": "Lubrication & oil management",
    "2.1": "Elastocaloric", "2.2": "Electrocaloric", "2.3": "Magnetocaloric", "2.4": "Ionocaloric", "2.5": "Barocaloric",
    "2.6": "Thermoelectric", "2.7": "Electro- & Chemisorption", "2.8": "Thermoacoustic", "2.9": "Sorption / Absorption", "2.10": "Hybrid & cascade",
    "3.1": "Topology & configuration", "3.2": "Controls & optimisation", "3.3": "Operational integration",
    "4.1": "Flexible cycles", "4.2": "Defrost & icing mitigation", "4.3": "Thermal storage",
    "5.1": "Design-for-disassembly", "5.2": "Modular assemblies", "5.3": "Recycled materials", "5.4": "Additive manufacturing", "5.5": "Predictive maintenance"
}

# ---- 2. KEYWORD UNIVERSE ---------------------------------------------------
KWS = {
    "1.1": ["scroll", "rotary", "vane", "twin-screw", "reciprocating", "isothermal compressor", "oil-free", "variable-speed", "economiser"],
    "1.2": ["R32", "R454B", "R290", "propane", "CO₂", "carbon dioxide", "low-GWP", "natural refrigerant", "azeotrope", "zeotropic", "ionic liquid"],
    "1.3": ["micro-channel", "plate heat exchanger", "fin-tube", "enthalpy exchanger", "phase-change heat exchanger", "anti-fouling", "frost-free", "3-D printed"],
    "1.4": ["IPM motor", "SiC inverter", "PMSM", "sensor-less", "flux-weakening"],
    "1.5": ["oil-separator", "mist injection", "CRII", "low-viscosity oil"],
    "2.1": ["elastocaloric", "shape-memory", "Ni-Ti"], "2.2": ["electrocaloric", "ferroelectric"], "2.3": ["magnetocaloric", "gadolinium"],
    "2.4": ["ionocaloric", "ion-solvation"], "2.5": ["barocaloric"], "2.6": ["thermoelectric", "Peltier", "Seebeck"],
    "2.7": ["electrochemical compressor", "chemisorption"], "2.8": ["thermoacoustic"],
    "2.9": ["adsorption heat pump", "absorption heat pump", "lithium bromide"], "2.10": ["cascade heat pump", "hybrid heat pump"],
    "3.1": ["cascade", "booster", "trans-critical", "bi-valent", "ground-source"], "3.2": ["model predictive control", "MPC", "PID", "fault detection", "digital twin"],
    "3.3": ["smart-grid", "thermal district network", "hybrid boiler"], "4.1": ["ejector cycle", "regenerative cycle", "parallel-compressor"],
    "4.2": ["defrost", "icing sensor", "hot-gas bypass", "nano-coating"], "4.3": ["PCM storage", "phase change material", "heat battery", "stratified tank"],
    "5.1": ["tool-less", "snap-fit", "reversible adhesive", "design-for-disassembly", "DfD"], "5.2": ["modular cartridge", "remanufacture", "refurbish", "life-extension"],
    "5.3": ["recycled", "bio-polymer", "PCR plastic", "reclaimed copper", "low-GWP foam"], "5.4": ["additive manufacturing", "3-D print", "near-net shape", "binder-jet"],
    "5.5": ["predictive maintenance", "condition monitoring", "service-as-a-product"]
}
KW2CAT = {kw.lower(): cat for cat, kwlist in KWS.items() for kw in kwlist}


In [ ]:
## 3 – ISO-59004 circularity heuristic
ISO_LEVERS = {
    "maintain": [
        "predictive maintenance", "preventive maintenance", "condition monitoring",
        "remote diagnostics", "fault detection", "service-as-a-product", "self-healing system"
    ],
    "reuse": [
        "remanufacture", "remanufacturing", "refurbish", "refurbished", "repair", "repairability",
        "cartridge", "replaceable module", "core exchange", "component reuse", "second life"
    ],
    "recycle": [
        "recycled", "recycling", "reclaim", "reclaimed", "material recovery",
        "mono-material", "material loop", "mechanical recycling", "chemical recycling", "closed-loop"
    ],
    "reduce": [
        "near-net shape", "additive manufacturing", "3d printing", "lightweight design",
        "material efficiency", "yield improvement", "low-waste", "minimal material use", "net-shape forming"
    ],
    "regenerate": [
        "bio-based", "biobased", "bio-polymer", "biopolymer", "biocomposite",
        "renewable feedstock", "natural material", "biodegradable", "biomaterial", "plant-derived"
    ]
}               

lemmatizer = WordNetLemmatizer()

def _norm(t):  return [lemmatizer.lemmatize(w) for w in re.findall(r"\b\w+\b", t.lower())]

def iso_score(text: str) -> int:
    toks = " ".join(_norm(text))
    hits = []
    for kws in ISO_LEVERS.values():
        m = sum(1 for kw in kws if kw in toks)
        hits.append(0 if m == 0 else 1 if m == 1 else 2 if m == 2 else 3)
    return math.ceil(sum(hits)/5)


In [ ]:
## 4 – System message for GPT
SYSTEM_MESSAGE = f"""
You are a sustainable-heating technology analyst. …
4. Score 0-3 for:
   a) cost_reduction_potential
   b) efficiency_gain_potential
   c) circularity_score (ISO 59004 lever heuristic)
Return ONLY JSON with the six fields requested.
"""


In [ ]:
## 5 – Structured output spec for batch_check
FIELDS = [
    {"name":"is_relevant","type":"str","description":"'yes' or 'no'"},
    {"name":"summary","type":"str","description":"≤25 words"},
    {"name":"category","type":"str","description":f"One of {list(CATEGORIES)}"},
    {"name":"cost_score","type":"int","description":"0-3"},
    {"name":"efficiency_score","type":"int","description":"0-3"},
    {"name":"circularity_score","type":"int","description":"0-3 (LLM view)"}
]


In [ ]:
## 6 – Create ID→text dicts (full datasets)
paper_dict  = openalex_df.set_index("id")["title_abstract"].to_dict()
patent_dict = patents_df .set_index("publication_number")["title_abstract"].to_dict()


In [ ]:
## 7 – Run GPT in batches *and* post-process ISO score
OUTPUT_DIR = PROJECT_DIR / "outputs"; OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def run_and_enrich(data, name, path, batch=30):
    proc = batch_check.LLMProcessor(
        model_name="gpt-4o-mini", temperature=0,
        system_message=SYSTEM_MESSAGE, session_name=name,
        output_fields=FIELDS, output_path=str(path)
    )
    proc.run(data, batch_size=batch, sleep_time=0.5)

    return


pat_out = run_and_enrich(
    {k:v for k,v in patent_dict.items() if any(w in v for w in KW2CAT)},
    "hp_patents_full", OUTPUT_DIR/"patents2_classified.jsonl"
)

pap_out = run_and_enrich(
    paper_dict, "hp_papers_full", OUTPUT_DIR/"papers2_classified.jsonl"
)


In [ ]:
path = '/home/pascualdiego/projects/DiscoveryHP/outputs/patents2_classified.jsonl'
df = pd.read_json(path, lines=True)